# 18 · Deep Learning Foundations

- **Level:** Advanced
- **Typical semester:** Semester 6–8 (Deep Learning, Neural Networks, Computer Vision)
- **Prerequisites:** 15, 16, 17
- **Matches code in:** `lipipy.dl`


## 1. Neuron and multilayer perceptron

$$a^{(l)} = \phi\left(W^{(l)}a^{(l-1)} + b^{(l)}\right), \qquad a^{(0)} = x$$

**Universal approximation theorem:** a one-hidden-layer network with a non-polynomial activation can approximate any continuous function on a compact set to arbitrary accuracy (given enough units).


## 2. Activation functions

| Name | $\phi(x)$ | $\phi'(x)$ |
|---|---|---|
| Sigmoid | $\frac{1}{1 + e^{-x}}$ | $\phi(1 - \phi)$ |
| Tanh | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $1 - \phi^2$ |
| ReLU | $\max(0, x)$ | $[x > 0]$ |
| Leaky ReLU | $\max(\alpha x, x)$ | $1$ or $\alpha$ |
| ELU | $x$ if $x>0$, else $\alpha(e^x - 1)$ | $1$ or $\phi + \alpha$ |
| SELU | $\lambda\,\text{ELU}_\alpha(x)$, $\lambda\approx1.0507$, $\alpha\approx1.6733$ | |
| Softplus | $\ln(1 + e^x)$ | $\sigma(x)$ |
| GELU | $x\Phi(x)\approx0.5x\left(1 + \tanh\left[\sqrt{2/\pi}(x + 0.044715x^3)\right]\right)$ | $\Phi(x) + x\varphi(x)$ |
| SiLU / Swish | $x\sigma(x)$ | $\sigma(x)(1 + x(1 - \sigma(x)))$ |
| Mish | $x\tanh(\text{softplus}(x))$ | |

**Gated linear units:** $\text{GLU}(x) = (xW)\odot\sigma(xV)$; $\;\text{SwiGLU}(x) = \text{SiLU}(xW)\odot(xV)$ (used in LLaMA-style FFNs)


## 3. Loss functions

MSE $\frac1n\sum(y - \hat y)^2$; MAE; Huber (see 17 §2)

Cross-entropy $-\sum_ky_k\log\hat p_k$; with label smoothing $y_k^{LS} = (1 - \varepsilon)y_k + \frac{\varepsilon}{K}$

**Focal loss:** $FL(p_t) = -\alpha_t(1 - p_t)^\gamma\log p_t$

Hinge $\max(0, 1 - y\hat y)$; KL divergence; negative log-likelihood

**Contrastive (pair) loss:** $y d^2 + (1 - y)\max(0, m - d)^2$. **Triplet loss:** $\max(0, \|a - p\|^2 - \|a - n\|^2 + m)$

**Dice loss:** $1 - \frac{2|A\cap B|}{|A| + |B|}$; IoU $= \frac{|A\cap B|}{|A\cup B|}$

**CTC loss:** $-\log\sum_{\pi\in\mathcal B^{-1}(y)}\prod_tp(\pi_t|x)$, computed with a forward-backward DP


## 4. Backpropagation

With pre-activation $z^{(l)} = W^{(l)}a^{(l-1)} + b^{(l)}$ and error $\delta^{(l)} = \frac{\partial L}{\partial z^{(l)}}$:
$$\delta^{(L)} = \nabla_aL\odot\phi'(z^{(L)}), \qquad \delta^{(l)} = \left(W^{(l+1)\top}\delta^{(l+1)}\right)\odot\phi'(z^{(l)})$$
$$\frac{\partial L}{\partial W^{(l)}} = \delta^{(l)}a^{(l-1)\top}, \qquad \frac{\partial L}{\partial b^{(l)}} = \delta^{(l)}$$

Vanishing/exploding gradients come from the product $\prod_l W^{(l)\top}\text{diag}(\phi'(z^{(l)}))$.


## 5. Initialization

**Xavier/Glorot:** $\text{Var}(W) = \frac{2}{n_{in} + n_{out}}$ (uniform $U\left[-\sqrt{\frac{6}{n_{in} + n_{out}}}, \sqrt{\frac{6}{n_{in}+n_{out}}}\right]$), for tanh/sigmoid

**He/Kaiming:** $\text{Var}(W) = \frac{2}{n_{in}}$, for ReLU

**LeCun:** $\text{Var}(W) = \frac1{n_{in}}$, for SELU

Residual branches are often scaled by $\frac{1}{\sqrt{2L}}$ (GPT-2) or initialized to zero.


## 6. Normalization layers

**Batch norm:** $\hat x = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$, $y = \gamma\hat x + \beta$, statistics over the batch; running averages at inference

**Layer norm:** same formula, statistics over the features of one sample

**RMSNorm:** $y = \frac{x}{\sqrt{\frac1d\sum x_i^2 + \epsilon}}\odot\gamma$

**Group norm, instance norm:** statistics over channel groups or single channels; **weight norm:** $w = g\frac{v}{\|v\|}$; **spectral norm:** $W/\sigma_{\max}(W)$

Pre-norm residual block: $x + F(\text{Norm}(x))$; post-norm: $\text{Norm}(x + F(x))$


## 7. Regularization

L2 / weight decay $\frac\lambda2\|w\|^2$; L1 $\lambda\|w\|_1$; max-norm constraint

**Dropout:** $\tilde a = \frac{m\odot a}{1 - p}$, $m_i\sim\text{Bernoulli}(1 - p)$ (inverted dropout, no change at test time)

Early stopping; data augmentation; **mixup:** $\tilde x = \lambda x_i + (1 - \lambda)x_j$, $\tilde y = \lambda y_i + (1-\lambda)y_j$, $\lambda\sim\text{Beta}(\alpha,\alpha)$; cutmix; stochastic depth


## 8. Convolutional networks

$$(I * K)(i,j) = \sum_m\sum_nI(i + m, j + n)K(m,n) \quad\text{(cross-correlation, as used in DL)}$$

**Output size:** $O = \left\lfloor\frac{W - K + 2P}{S}\right\rfloor + 1$; with dilation $d$, effective kernel $K_{eff} = d(K - 1) + 1$

**Parameters:** $(K_h K_w C_{in} + 1)C_{out}$; FLOPs $\approx 2K_hK_wC_{in}C_{out}H_{out}W_{out}$

**Receptive field:** $r_l = r_{l-1} + (K_l - 1)\prod_{i<l}S_i$

**Depthwise-separable conv** cost ratio vs standard: $\frac1{C_{out}} + \frac1{K^2}$

**Transposed conv output:** $O = (W - 1)S - 2P + K$ (+ output padding)

**Residual block (ResNet):** $y = x + F(x)$, so $\frac{\partial y}{\partial x} = I + \frac{\partial F}{\partial x}$ and gradients flow through the identity

**Pooling:** max, average, global average; **squeeze-and-excitation:** $s = \sigma(W_2\,\text{ReLU}(W_1\,\text{GAP}(x)))$, $y = s\odot x$

**EfficientNet compound scaling:** depth $\alpha^\phi$, width $\beta^\phi$, resolution $\gamma^\phi$ with $\alpha\beta^2\gamma^2\approx2$


## 9. Object detection and segmentation

IoU (see §3); non-maximum suppression; anchor box regression $t_x = \frac{x - x_a}{w_a}$, $t_w = \log\frac{w}{w_a}$

mAP = mean over classes of the area under the precision-recall curve

YOLO loss = localization + objectness + classification terms. DETR uses Hungarian matching between predictions and ground truth.


## 10. Recurrent networks

**Vanilla RNN:** $h_t = \tanh(W_hh_{t-1} + W_xx_t + b)$, $y_t = W_yh_t$

**LSTM:**
$$f_t = \sigma(W_f[h_{t-1}, x_t] + b_f), \quad i_t = \sigma(W_i[h_{t-1},x_t] + b_i), \quad o_t = \sigma(W_o[h_{t-1},x_t] + b_o)$$
$$\tilde c_t = \tanh(W_c[h_{t-1}, x_t] + b_c), \quad c_t = f_t\odot c_{t-1} + i_t\odot\tilde c_t, \quad h_t = o_t\odot\tanh(c_t)$$

**GRU:**
$$z_t = \sigma(W_z[h_{t-1}, x_t]), \quad r_t = \sigma(W_r[h_{t-1}, x_t]), \quad \tilde h_t = \tanh(W[r_t\odot h_{t-1}, x_t]), \quad h_t = (1 - z_t)\odot h_{t-1} + z_t\odot\tilde h_t$$

**Bahdanau attention (seq2seq):** $e_{tj} = v^\top\tanh(W_sh_{t-1}^{dec} + W_hh_j^{enc})$, $\alpha_{tj} = \text{softmax}_j(e_{tj})$, context $c_t = \sum_j\alpha_{tj}h_j^{enc}$


## 11. Autoencoders and embeddings

Autoencoder: $\min\|x - g(f(x))\|^2$; denoising: $\min\|x - g(f(x + \epsilon))\|^2$; sparse: $+\lambda\sum_j KL(\rho\|\hat\rho_j)$; contractive: $+\lambda\|J_f(x)\|_F^2$

**Word2Vec skip-gram with negative sampling:** $\log\sigma(u_o^\top v_c) + \sum_{k=1}^K\mathbb E_{w_k\sim P_n}\log\sigma(-u_{w_k}^\top v_c)$, $P_n(w)\propto\text{count}(w)^{3/4}$

**GloVe:** $\sum_{ij}f(X_{ij})(w_i^\top\tilde w_j + b_i + \tilde b_j - \log X_{ij})^2$


## 12. Training at scale: practical formulas

Parameters of a dense layer: $d_{in}d_{out} + d_{out}$. Memory for Adam training in mixed precision $\approx 16$ bytes/parameter (weights, gradients, two moments, master copy) plus activations.

**Knowledge distillation:** $L = (1 - \alpha)\,CE(y, p_s) + \alpha T^2KL\left(\text{softmax}(z_t/T)\,\|\,\text{softmax}(z_s/T)\right)$

**Quantization:** $q = \text{round}\left(\frac{x}{s}\right) + z$, $\;\hat x = s(q - z)$, $\;s = \frac{x_{\max} - x_{\min}}{2^b - 1}$

**Magnitude pruning:** keep weights with $|w|$ above a threshold; **lottery ticket hypothesis:** a sparse subnetwork, rewound to its initial weights, can train to full accuracy.
